In [0]:
from pyspark.sql import functions as F

LANDING    = "/Volumes/olist/bronze/landing"
CHECKPOINT = "/Volumes/olist/bronze/landing/_checkpoints"

TABLES = [
    "orders", "order_items", "order_payments", "order_reviews",
    "customers", "sellers", "products", "category_translation",
]

def ingest(name: str):
    (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/{name}/schema")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("header", "true")
        .load(f"{LANDING}/{name}")
        .select(
            "*",
            F.col("_metadata.file_name").alias("_source_file"),
            F.current_timestamp().alias("_ingested_at"),
        )
        .writeStream
        .option("checkpointLocation", f"{CHECKPOINT}/{name}/write")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"olist.bronze.{name}")
        .awaitTermination())

for t in TABLES:
    print(f"ingesting {t} ...")
    ingest(t)
    print(f"  done")

In [0]:
%sql
SELECT 'orders' AS t, COUNT(*) FROM olist.bronze.orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist.bronze.order_items
UNION ALL SELECT 'customers', COUNT(*) FROM olist.bronze.customers
UNION ALL SELECT 'order_reviews', COUNT(*) FROM olist.bronze.order_reviews;

In [0]:
%sql
SELECT 'orders' AS t, COUNT(*) FROM olist.bronze.orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist.bronze.order_items
UNION ALL SELECT 'customers', COUNT(*) FROM olist.bronze.customers
UNION ALL SELECT 'order_reviews', COUNT(*) FROM olist.bronze.order_reviews;

In [0]:
%sql
SELECT COUNT(*) AS malformed
FROM olist.bronze.order_reviews
WHERE review_id IS NULL OR LENGTH(review_id) <> 32;

In [0]:
def ingest(name: str, extra_options: dict = None):
    reader = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/{name}/schema")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("header", "true"))

    for k, v in (extra_options or {}).items():
        reader = reader.option(k, v)

    (reader.load(f"{LANDING}/{name}")
        .select("*",
                F.col("_metadata.file_name").alias("_source_file"),
                F.current_timestamp().alias("_ingested_at"))
        .writeStream
        .option("checkpointLocation", f"{CHECKPOINT}/{name}/write")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"olist.bronze.{name}")
        .awaitTermination())

MULTILINE = {"multiLine": "true", "escape": '"'}

for t in TABLES:
    opts = MULTILINE if t == "order_reviews" else None
    print(f"ingesting {t} ...")
    ingest(t, opts)

In [0]:
spark.sql("DROP TABLE IF EXISTS olist.bronze.order_reviews")
dbutils.fs.rm(f"{CHECKPOINT}/order_reviews", recurse=True)
ingest("order_reviews", MULTILINE)

In [0]:
%sql
SELECT COUNT(*) AS total,
       SUM(CASE WHEN review_id IS NULL OR LENGTH(review_id) <> 32 THEN 1 ELSE 0 END) AS malformed
FROM olist.bronze.order_reviews;

In [0]:
%sql
SELECT 'orders' AS t, COUNT(*) FROM olist.bronze.orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist.bronze.order_items
UNION ALL SELECT 'customers', COUNT(*) FROM olist.bronze.customers
UNION ALL SELECT 'order_reviews', COUNT(*) FROM olist.bronze.order_reviews;